# LeetCode 3146 — Permutation Difference between Two Strings bằng Python

Link đề: https://leetcode.com/problems/permutation-difference-between-two-strings/ · Độ khó: **Easy** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def findPermutationDifference(self, s: str, t: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `t` là một **hoán vị** của `s` (mỗi ký tự trong `s` xuất hiện **đúng một lần**); **độ chênh lệch hoán vị** là **tổng**, trên mọi ký tự, của **|vị trí của ký tự trong `s` − vị trí của ký tự đó trong `t`|**.
2. Nắm các công cụ Python: `str.index(c)` (vị trí đầu tiên của ký tự), **`dict` lưu vị trí** (`pos[c] = i`), `enumerate`, `abs`, `sum`, và mảng đếm `26` phần tử với `ord(c) - ord('a')`.
3. Đi từ cách **hỏi vị trí của từng ký tự bằng `index`** → **dùng `dict` ghi sẵn vị trí trong `t`** → **hai `dict` vị trí rồi so từng ký tự** → **mảng `26` phần tử thay `dict`**.
4. Hiểu mấu chốt: ta cần **tra vị trí của một ký tự trong `t` nhanh** (`O(1)`), thay vì quét `t` cho mỗi ký tự; một lượt ghi vị trí rồi một lượt cộng là đủ.
5. Tránh các lỗi kinh điển: so sánh `s[i]` với `t[i]` **theo vị trí** (thay vì theo ký tự), **quên `abs`** (các hiệu âm/dương triệt tiêu: tổng luôn bằng `0`), tính trùng một ký tự hai lần, và dùng `index` khi chuỗi có ký tự lặp (đề này đảm bảo không lặp).

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `t` sao cho **mỗi ký tự xuất hiện nhiều nhất một lần trong `s`** và **`t` là một hoán vị của `s`** (cùng các ký tự, khác thứ tự).

**Độ chênh lệch hoán vị** giữa `s` và `t` được định nghĩa là **tổng** của **hiệu tuyệt đối giữa chỉ số của ký tự đó trong `s` và chỉ số của ký tự đó trong `t`**, cộng trên **mọi ký tự**.

Hãy trả về độ chênh lệch hoán vị giữa `s` và `t`.

**Ví dụ 1**

- Input: `s = "abc"`, `t = "bac"`
- Output: `2`
- Giải thích: ký tự `a`: chỉ số `0` trong `s`, `1` trong `t` ⇒ `|0 − 1| = 1`; ký tự `b`: `1` trong `s`, `0` trong `t` ⇒ `|1 − 0| = 1`; ký tự `c`: `2` và `2` ⇒ `0`. Tổng `1 + 1 + 0 = 2`.

**Ví dụ 2**

- Input: `s = "abcde"`, `t = "edbac"`
- Output: `12`
- Giải thích: `a`: `|0 − 3| = 3`; `b`: `|1 − 2| = 1`; `c`: `|2 − 4| = 2`; `d`: `|3 − 1| = 2`; `e`: `|4 − 0| = 4`. Tổng `3 + 1 + 2 + 2 + 4 = 12`.

**Ràng buộc:** `1 <= s.length <= 26`; mỗi ký tự xuất hiện nhiều nhất một lần trong `s`; `t` là hoán vị của `s`; `s` và `t` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import string

examples = [
    ("abc", "bac", 2),
    ("abcde", "edbac", 12),
    ("a", "a", 0),
]
for s, t, expected in examples:
    print(repr(s), repr(t), "-> kỳ vọng", expected)

'abc' 'bac' -> kỳ vọng 2
'abcde' 'edbac' -> kỳ vọng 12
'a' 'a' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Vị trí của ký tự: `str.index`

`s.index(c)` trả về **chỉ số của lần xuất hiện đầu tiên** của ký tự `c` trong `s` (chỉ số từ `0`). Nếu `c` không có thì báo `ValueError`. Với chuỗi **không có ký tự lặp** (như đề này) chỉ số đó là **duy nhất**. Mỗi lần gọi `index` quét chuỗi từ đầu nên tốn `O(n)`.

In [2]:
s = "abcde"
print(s.index("a"), s.index("c"), s.index("e"))     # 0 2 4
try:
    s.index("z")
except ValueError as e:
    print("ValueError:", e)

0 2 4
ValueError: substring not found


### 3.2. Từ điển (dict) lưu vị trí: `pos[c] = i`

Thay vì tìm vị trí mỗi lần, ta **ghi sẵn** vào `dict`: khóa là ký tự, giá trị là chỉ số. Dùng `enumerate(s)` để lặp cùng lúc chỉ số và ký tự. Sau đó `pos[c]` cho vị trí trong `O(1)` (trung bình). Có thể viết bằng dict comprehension: `{c: i for i, c in enumerate(s)}`.

In [3]:
t = "edbac"
pos_t = {}
for i, c in enumerate(t):
    pos_t[c] = i
print(pos_t)                                   # {'e': 0, 'd': 1, 'b': 2, 'a': 3, 'c': 4}
print({c: i for i, c in enumerate(t)} == pos_t)    # True
print(pos_t["a"])                              # 3

{'e': 0, 'd': 1, 'b': 2, 'a': 3, 'c': 4}
True
3


### 3.3. `abs` và `sum` trên các cặp

`abs(x)` là giá trị tuyệt đối. `sum(biểu_thức for ...)` cộng các số. Tổng độ chênh lệch là `sum(abs(i - pos_t[c]) for i, c in enumerate(s))`.

In [4]:
s, t = "abc", "bac"
pos_t = {c: i for i, c in enumerate(t)}
parts = [abs(i - pos_t[c]) for i, c in enumerate(s)]
print(parts, sum(parts))                       # [1, 1, 0] 2

[1, 1, 0] 2


### 3.4. Mảng 26 phần tử thay cho dict

Vì chỉ có chữ thường, ta có thể dùng list 26 số: `pos[ord(c) - ord('a')] = i`. `ord('a') = 97`, nên `ord(c) - 97` là `0..25`. Mảng nhanh hơn dict một chút và không phụ thuộc băm.

In [5]:
pos = [-1] * 26
for i, c in enumerate("edbac"):
    pos[ord(c) - ord("a")] = i
print(pos[:5])                                 # [3, 2, 4, 1, 0] -> vị trí của a, b, c, d, e trong t

[3, 2, 4, 1, 0]


### 3.5. Hiệu có dấu và `abs`: vì sao `abs` quan trọng

Với hoán vị, **tổng các hiệu có dấu** `Σ (i − j)` luôn bằng `0` (vì tổng các chỉ số của `s` và của `t` đều bằng `0 + 1 + ... + (n−1)`). Do đó **không có `abs` ta luôn ra `0`**. Cần `abs` cho **từng** ký tự trước khi cộng.

In [6]:
s, t = "abcde", "edbac"
pos_t = {c: i for i, c in enumerate(t)}
signed = [i - pos_t[c] for i, c in enumerate(s)]
print(signed, sum(signed))                      # [-3, -1, -2, 2, 4] 0  (triệt tiêu)
print([abs(x) for x in signed], sum(abs(x) for x in signed))   # [3, 1, 2, 2, 4] 12

[-3, -1, -2, 2, 4] 0
[3, 1, 2, 2, 4] 12


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "abcde"`, `t = "edbac"`, ta viết **vị trí của từng ký tự** trong hai chuỗi:

| ký tự | vị trí trong `s` | vị trí trong `t` | `|hiệu|` |
|---|---|---|---|
| a | 0 | 3 | 3 |
| b | 1 | 2 | 1 |
| c | 2 | 4 | 2 |
| d | 3 | 1 | 2 |
| e | 4 | 0 | 4 |

Tổng `3 + 1 + 2 + 2 + 4 = 12`.

Mỗi ký tự **được so với chính nó** ở chuỗi kia (theo **ký tự**, không phải theo vị trí). Nhìn kĩ: nếu ta so `s[i]` với `t[i]` theo vị trí `i` (a-e, b-d, c-b, d-a, e-c) thì hoàn toàn không liên quan tới đề.

In [7]:
s, t = "abcde", "edbac"
for c in s:
    print(f"{c}: vị trí trong s = {s.index(c)}, trong t = {t.index(c)}, |hiệu| = {abs(s.index(c) - t.index(c))}")

a: vị trí trong s = 0, trong t = 3, |hiệu| = 3
b: vị trí trong s = 1, trong t = 2, |hiệu| = 1
c: vị trí trong s = 2, trong t = 4, |hiệu| = 2
d: vị trí trong s = 3, trong t = 1, |hiệu| = 2
e: vị trí trong s = 4, trong t = 0, |hiệu| = 4


## 5. Cách 1 — Brute Force / baseline: hỏi `index` cho từng ký tự

**Ý tưởng.** Với mỗi ký tự `c` của `s`: vị trí trong `s` là `s.index(c)`, vị trí trong `t` là `t.index(c)`; cộng `abs` của hiệu. Một dòng, bám sát đề.

In [8]:
def permutation_difference_index(s: str, t: str) -> int:
    total = 0
    for c in s:
        total += abs(s.index(c) - t.index(c))       # vị trí của c trong s và trong t
    return total


for s, t, expected in examples:
    print(permutation_difference_index(s, t), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
12 (kỳ vọng 12 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)` (`n <= 26`). Mỗi ký tự gọi `index` hai lần, mỗi lần `O(n)` ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 26` là tối đa `~1300` bước, nhanh.

**Nút thắt:** `index` quét lại chuỗi mỗi lần. Ta có thể quét `t` **một lần** để ghi sẵn vị trí của mọi ký tự.

## 6. Ý tưởng thuật toán / bước đột phá

**Ghi sẵn vị trí.** Duyệt `t` một lần, lưu `pos_t[c] = chỉ số`. Sau đó duyệt `s` một lần (kèm chỉ số `i`), cộng `abs(i − pos_t[c])`. Mỗi ký tự được tra trong `O(1)`; tổng `O(n)`.

Biến thể đối xứng: lưu **cả hai** `pos_s` và `pos_t` rồi cộng `abs(pos_s[c] − pos_t[c])` cho mọi `c` thuộc `pos_s`. Hoặc dùng mảng 26 phần tử thay dict. Tất cả đều `O(n)`.

In [9]:
s, t = "abcde", "edbac"
pos_t = {c: i for i, c in enumerate(t)}
total = 0
for i, c in enumerate(s):
    diff = abs(i - pos_t[c])
    total += diff
    print(f"i={i} c={c}: pos_t[{c}]={pos_t[c]}, |{i} - {pos_t[c]}| = {diff}, tổng = {total}")

i=0 c=a: pos_t[a]=3, |0 - 3| = 3, tổng = 3
i=1 c=b: pos_t[b]=2, |1 - 2| = 1, tổng = 4
i=2 c=c: pos_t[c]=4, |2 - 4| = 2, tổng = 6
i=3 c=d: pos_t[d]=1, |3 - 1| = 2, tổng = 8
i=4 c=e: pos_t[e]=0, |4 - 0| = 4, tổng = 12


## 7. Các cách tiếp theo

### Cách 2 — Một `dict` vị trí của `t`, rồi duyệt `s`

**Giải thích từng đoạn.**

1. `pos_t = {c: i for i, c in enumerate(t)}`: vị trí mỗi ký tự trong `t`.
2. `for i, c in enumerate(s)`: `i` là vị trí trong `s`.
3. `total += abs(i - pos_t[c])`.

In [10]:
def permutation_difference_dict(s: str, t: str) -> int:
    pos_t = {c: i for i, c in enumerate(t)}                       # vị trí của mỗi ký tự trong t
    return sum(abs(i - pos_t[c]) for i, c in enumerate(s))        # i: vị trí trong s


for s, t, expected in examples:
    print(permutation_difference_dict(s, t), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
12 (kỳ vọng 12 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDict:
    def findPermutationDifference(self, s: str, t: str) -> int:
        pos_t = {c: i for i, c in enumerate(t)}
        return sum(abs(i - pos_t[c]) for i, c in enumerate(s))


print(SolutionDict().findPermutationDifference("abcde", "edbac"))      # 12

12


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (hoặc `O(26)` vì chỉ có tối đa 26 ký tự khác nhau).

### Cách 3 — Hai `dict` vị trí rồi so từng ký tự

**Giải thích.** Lưu `pos_s` và `pos_t`; rồi cộng `abs(pos_s[c] − pos_t[c])` với mọi `c` trong `pos_s`. Cách này **đối xứng** (không phân biệt chuỗi nào là gốc) và hợp khi cần dùng lại `pos_s`. Lưu ý: nếu `t` thật sự là hoán vị của `s` thì hai dict có cùng tập khóa.

In [12]:
def permutation_difference_two_dicts(s: str, t: str) -> int:
    pos_s = {c: i for i, c in enumerate(s)}
    pos_t = {c: i for i, c in enumerate(t)}
    return sum(abs(pos_s[c] - pos_t[c]) for c in pos_s)


for s, t, expected in examples:
    print(permutation_difference_two_dicts(s, t), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
12 (kỳ vọng 12 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (hai dict).

### Cách 4 — Mảng 26 phần tử

**Giải thích.** Chỉ có chữ thường, nên dùng `pos = [0] * 26`: duyệt `t` ghi `pos[ord(c) - 97] = i`; duyệt `s` cộng `abs(i - pos[ord(c) - 97])`. Không cần `dict`, hằng số nhỏ.

In [13]:
def permutation_difference_array(s: str, t: str) -> int:
    pos = [0] * 26                                       # pos[k] = vị trí của chữ cái thứ k trong t
    for i, c in enumerate(t):
        pos[ord(c) - ord("a")] = i
    total = 0
    for i, c in enumerate(s):
        total += abs(i - pos[ord(c) - ord("a")])
    return total


for s, t, expected in examples:
    print(permutation_difference_array(s, t), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
12 (kỳ vọng 12 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(26) = O(1)`**.

### Cách 5 — Vòng lặp một lượt không dùng cấu trúc dữ liệu phụ (chỉ để so sánh)

**Giải thích.** Với `n <= 26` bạn có thể viết hai vòng lồng (không `dict`, không `index`): với mỗi `i` trong `s`, tìm `j` trong `t` sao cho `t[j] == s[i]` rồi cộng `abs(i - j)`. Cách này là phiên bản "khai triển" của `index` và cho thấy `index` làm gì bên trong.

In [14]:
def permutation_difference_nested(s: str, t: str) -> int:
    total = 0
    for i in range(len(s)):
        for j in range(len(t)):
            if s[i] == t[j]:                 # tìm vị trí của s[i] trong t
                total += abs(i - j)
                break
    return total


for s, t, expected in examples:
    print(permutation_difference_nested(s, t), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
12 (kỳ vọng 12 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(n²)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (dict) trên `s = "abcde"`, `t = "edbac"` và Cách 4 (mảng 26) trên `s = "abc"`, `t = "bac"`.

In [15]:
def trace_dict(s: str, t: str) -> int:
    pos_t = {}
    print(f"s = {s!r}, t = {t!r}")
    for i, c in enumerate(t):
        pos_t[c] = i
        print(f"  ghi vị trí trong t: pos_t[{c!r}] = {i}")
    total = 0
    for i, c in enumerate(s):
        diff = abs(i - pos_t[c])
        total += diff
        print(f"  ký tự {c!r}: vị trí s = {i}, vị trí t = {pos_t[c]}, |hiệu| = {diff}, tổng = {total}")
    return total


assert trace_dict("abcde", "edbac") == 12

s = 'abcde', t = 'edbac'
  ghi vị trí trong t: pos_t['e'] = 0
  ghi vị trí trong t: pos_t['d'] = 1
  ghi vị trí trong t: pos_t['b'] = 2
  ghi vị trí trong t: pos_t['a'] = 3
  ghi vị trí trong t: pos_t['c'] = 4
  ký tự 'a': vị trí s = 0, vị trí t = 3, |hiệu| = 3, tổng = 3
  ký tự 'b': vị trí s = 1, vị trí t = 2, |hiệu| = 1, tổng = 4
  ký tự 'c': vị trí s = 2, vị trí t = 4, |hiệu| = 2, tổng = 6
  ký tự 'd': vị trí s = 3, vị trí t = 1, |hiệu| = 2, tổng = 8
  ký tự 'e': vị trí s = 4, vị trí t = 0, |hiệu| = 4, tổng = 12


In [16]:
def trace_array(s: str, t: str) -> int:
    pos = [0] * 26
    for i, c in enumerate(t):
        pos[ord(c) - 97] = i
    print("pos (a..e) sau khi duyệt t:", pos[:5])
    total = 0
    for i, c in enumerate(s):
        k = ord(c) - 97
        total += abs(i - pos[k])
        print(f"i={i} c={c!r} (k={k}): |{i} - {pos[k]}| -> tổng = {total}")
    return total


assert trace_array("abc", "bac") == 2

pos (a..e) sau khi duyệt t: [1, 0, 2, 0, 0]
i=0 c='a' (k=0): |0 - 1| -> tổng = 1
i=1 c='b' (k=1): |1 - 0| -> tổng = 2
i=2 c='c' (k=2): |2 - 2| -> tổng = 2


Bảng trace Cách 2 với `s = "abcde"`, `t = "edbac"`:

| i (vị trí trong `s`) | ký tự | `pos_t[ký tự]` | `|i − pos_t|` | tổng |
|---|---|---|---|---|
| 0 | a | 3 | 3 | 3 |
| 1 | b | 2 | 1 | 4 |
| 2 | c | 4 | 2 | 6 |
| 3 | d | 1 | 2 | 8 |
| 4 | e | 0 | 4 | 12 |

Đáp án `12`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (một `dict` vị trí của `t`): ngắn, `O(n)`.

In [17]:
class Solution:
    def findPermutationDifference(self, s: str, t: str) -> int:
        pos_t = {c: i for i, c in enumerate(t)}                      # vị trí của mỗi ký tự trong t
        return sum(abs(i - pos_t[c]) for i, c in enumerate(s))       # i là vị trí của c trong s


sol = Solution()
for s, t, expected in examples:
    print(repr(s), repr(t), "->", sol.findPermutationDifference(s, t))

'abc' 'bac' -> 2
'abcde' 'edbac' -> 12
'a' 'a' -> 0


## 10. Vì sao thuật toán đúng

- **Vị trí duy nhất:** mỗi ký tự xuất hiện đúng một lần trong `s`, và `t` là hoán vị của `s` nên mỗi ký tự cũng xuất hiện đúng một lần trong `t`. Do đó `pos_t[c]` xác định duy nhất (không bị ghi đè bởi lần xuất hiện khác).
- **Mọi ký tự có trong `pos_t`:** vì `t` là hoán vị của `s`, mọi `c` của `s` đều có trong `t`, nên `pos_t[c]` không gây `KeyError`.
- **Đúng công thức:** với mỗi `c` ở vị trí `i` trong `s`, đóng góp `|i − pos_t[c]|`; cộng theo `enumerate(s)` là cộng trên **mọi** ký tự đúng một lần.
- **Cần `abs` từng phần:** tổng các hiệu có dấu luôn `0` (cả hai chuỗi đều dùng đủ các chỉ số `0..n−1`), nên `abs` phải áp dụng cho **từng** ký tự.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`n <= 26`).

| Cách | Time | Space |
|---|---|---|
| 1. `s.index` / `t.index` | `O(n²)` | `O(1)` |
| 2. Một `dict` vị trí | `O(n)` | `O(n)` |
| 3. Hai `dict` vị trí | `O(n)` | `O(n)` |
| 4. Mảng 26 phần tử | `O(n)` | `O(1)` |
| 5. Hai vòng lồng | `O(n²)` | `O(1)` |

Với `n <= 26` mọi cách đều tức thì.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "index": permutation_difference_index,
    "dict": permutation_difference_dict,
    "two_dicts": permutation_difference_two_dicts,
    "array": permutation_difference_array,
    "nested": permutation_difference_nested,
    "leetcode": sol.findPermutationDifference,
}


def check_all(s, t, expected, label):
    for key, fn in impls.items():
        actual = fn(s, t)
        assert actual == expected, (label, key, s, t, actual, expected)
    print("PASS", label)


for idx, (s, t, expected) in enumerate(examples, 1):
    check_all(s, t, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("a", "a", 0, "một ký tự")
check_all("ab", "ab", 0, "hai ký tự, giống nhau")
check_all("ab", "ba", 2, "hai ký tự đảo nhau: 1 + 1")
check_all("abc", "abc", 0, "giống hệt nhau")
check_all("abc", "cba", 4, "đảo ngược ba ký tự: 2 + 0 + 2")
check_all("abc", "bca", 4, "xoay vòng: 2 + 1 + 1")
check_all("abc", "cab", 4, "xoay vòng ngược")
check_all("abcd", "badc", 4, "hoán đổi từng cặp: 1 + 1 + 1 + 1")
alpha = string.ascii_lowercase
check_all(alpha, alpha, 0, "26 ký tự giống nhau")
check_all(alpha, alpha[::-1], sum(abs(i - (25 - i)) for i in range(26)), "26 ký tự đảo ngược")
check_all("zyx", "xyz", 4, "chữ cái cuối bảng")

PASS một ký tự
PASS hai ký tự, giống nhau
PASS hai ký tự đảo nhau: 1 + 1
PASS giống hệt nhau
PASS đảo ngược ba ký tự: 2 + 0 + 2
PASS xoay vòng: 2 + 1 + 1
PASS xoay vòng ngược
PASS hoán đổi từng cặp: 1 + 1 + 1 + 1
PASS 26 ký tự giống nhau
PASS 26 ký tự đảo ngược
PASS chữ cái cuối bảng


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **so sánh `s[i]` với `t[i]` theo vị trí** (so mã ký tự thay vì vị trí); (b) **quên `abs`** (tổng luôn `0`).

In [20]:
def wrong_positionwise(s, t):
    return sum(abs(ord(a) - ord(b)) for a, b in zip(s, t))       # SAI: so mã ký tự theo vị trí


def wrong_no_abs(s, t):
    pos_t = {c: i for i, c in enumerate(t)}
    return sum(i - pos_t[c] for i, c in enumerate(s))            # SAI: thiếu abs


print(wrong_positionwise("ace", "eac"), "!=", permutation_difference_dict("ace", "eac"))
print(wrong_no_abs("abcde", "edbac"), "!=", permutation_difference_dict("abcde", "edbac"))
assert wrong_positionwise("ace", "eac") != permutation_difference_dict("ace", "eac")
assert wrong_no_abs("abcde", "edbac") != permutation_difference_dict("abcde", "edbac")

8 != 4
0 != 12


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn hoán vị

Đầu vào hợp lệ: `s` là các chữ cái **khác nhau** (độ dài `1..26`), `t` là **hoán vị** của `s`. Ta kiểm tra: (1) **ngẫu nhiên** với `n` tới `26`; (2) **vét cạn mọi hoán vị** của `"abcde"` (`5! = 120`). Thêm thuộc tính: độ chênh lệch luôn **chẵn** (tổng các `|hiệu|` bằng hai lần tổng các hiệu dương, vì tổng hiệu có dấu bằng `0`).

In [21]:
import itertools

rng = random.Random(3146)
for trial in range(3000):
    n = rng.choice([1, 2, 3, 5, 10, 25, 26])
    letters = rng.sample(string.ascii_lowercase, n)               # n chữ cái khác nhau
    s = "".join(letters)
    t_list = list(letters)
    rng.shuffle(t_list)
    t = "".join(t_list)
    assert 1 <= len(s) <= 26 and len(set(s)) == len(s) and sorted(s) == sorted(t)
    ref = permutation_difference_index(s, t)
    for key, fn in impls.items():
        assert fn(s, t) == ref, (key, s, t)
    assert ref % 2 == 0, "độ chênh lệch hoán vị luôn chẵn"
print("PASS 3000 cặp (s, t) ngẫu nhiên đúng ràng buộc")

count = 0
for perm in itertools.permutations("abcde"):
    t = "".join(perm)
    ref = permutation_difference_index("abcde", t)
    for key, fn in impls.items():
        assert fn("abcde", t) == ref, (key, t)
    count += 1
print("PASS vét cạn", count, "hoán vị của 'abcde'")

PASS 3000 cặp (s, t) ngẫu nhiên đúng ràng buộc
PASS vét cạn 120 hoán vị của 'abcde'


## 13. Các lỗi người mới hay gặp

### Lỗi 1: So sánh theo vị trí `s[i]` với `t[i]` (thay vì theo ký tự)

In [22]:
def sai_so_theo_vi_tri(s, t):
    total = 0
    for i in range(len(s)):
        if s[i] != t[i]:
            total += 1                  # SAI: đếm số vị trí khác nhau, không phải tổng khoảng cách vị trí
    return total


print("sai:", sai_so_theo_vi_tri("abcde", "edbac"), "| đúng:", Solution().findPermutationDifference("abcde", "edbac"))

sai: 5 | đúng: 12


**Vì sao sai:** đề yêu cầu tổng **khoảng cách vị trí của cùng một ký tự** trong hai chuỗi. Đếm số vị trí `s[i] != t[i]` chỉ cho biết bao nhiêu chỗ khác, không đo xa bao nhiêu. **Cách sửa:** với mỗi ký tự `c` tìm vị trí trong `s` và `t` rồi cộng `abs`.

### Lỗi 2: Quên `abs`

In [23]:
def sai_quen_abs(s, t):
    pos_t = {c: i for i, c in enumerate(t)}
    return sum(i - pos_t[c] for i, c in enumerate(s))


print("sai:", sai_quen_abs("abcde", "edbac"), "| đúng:", Solution().findPermutationDifference("abcde", "edbac"))
print("sai:", sai_quen_abs("abc", "bac"), "| đúng:", Solution().findPermutationDifference("abc", "bac"))

sai: 0 | đúng: 12
sai: 0 | đúng: 2


**Vì sao sai:** các hiệu có dấu triệt tiêu: với một hoán vị, `Σ (vị trí trong s) = Σ (vị trí trong t) = 0 + 1 + ... + (n − 1)`, nên tổng hiệu có dấu luôn `0`. **Cách sửa:** `abs` cho **từng** ký tự trước khi cộng.

### Lỗi 3: Cộng mỗi ký tự hai lần

In [24]:
def sai_cong_hai_lan(s, t):
    pos_s = {c: i for i, c in enumerate(s)}
    pos_t = {c: i for i, c in enumerate(t)}
    total = 0
    for c in s:
        total += abs(pos_s[c] - pos_t[c])
    for c in t:                                           # SAI: duyệt lần nữa -> mỗi ký tự cộng 2 lần
        total += abs(pos_s[c] - pos_t[c])
    return total


print("sai:", sai_cong_hai_lan("abc", "bac"), "| đúng:", Solution().findPermutationDifference("abc", "bac"))

sai: 4 | đúng: 2


**Vì sao sai:** `s` và `t` có **cùng** tập ký tự; duyệt cả hai làm mỗi ký tự được tính hai lần (kết quả `4` thay vì `2`). **Cách sửa:** chỉ duyệt **một** chuỗi (mỗi ký tự đúng một lần).

### Lỗi 4: Dùng `index` khi chuỗi có ký tự lặp (ngoài ràng buộc đề)

In [25]:
def dung_index(s, t):
    return sum(abs(s.index(c) - t.index(c)) for c in s)


# 'aab' có ký tự lặp: index luôn trả vị trí xuất hiện đầu tiên của 'a'
print(dung_index("aab", "baa"))      # kết quả không đáng tin vì 'a' lặp (đề đảm bảo KHÔNG lặp)
print("kiểm tra điều kiện đề:", len(set("abc")) == len("abc"), len(set("aab")) == len("aab"))

4
kiểm tra điều kiện đề: True False


**Lưu ý:** đề **đảm bảo** mỗi ký tự trong `s` xuất hiện nhiều nhất một lần, nên `index` cho vị trí duy nhất. Nếu bạn mở rộng bài sang chuỗi có ký tự lặp, định nghĩa "vị trí của ký tự" không còn rõ ràng (cần ghép các lần xuất hiện thứ nhất, thứ hai, ...). Đây không phải lỗi trong bài này, nhưng là điều cần nhớ khi biến thể.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Vị trí của cùng một phần tử trong hai dãy**" ⇒ lưu `pos[x] = i` trong `dict` rồi tra, thay vì `index` lặp lại.
- Với bảng chữ cái nhỏ (`26`), dùng **mảng** `ord(c) - ord('a')` thay `dict`.
- Tổng hiệu có dấu của hoán vị luôn bằng `0`: nhớ **`abs` từng phần** (hoặc dùng bình phương nếu bài khác).
- `enumerate` cho cả chỉ số và giá trị: tiện cho `pos = {c: i for i, c in enumerate(t)}`.
- Kiểm tra: `s` và `t` phải cùng tập ký tự (`sorted(s) == sorted(t)`) — hữu ích khi tự sinh test.

## 15. Biến thể của bài

1. **Độ chênh lệch bình phương** (`Σ (i − j)²`).
2. **Số lần đảo vị trí (inversions)** giữa `s` và `t` (khác với tổng khoảng cách).
3. **Chuỗi có ký tự lặp:** ghép lần xuất hiện thứ `k` của ký tự trong `s` với lần thứ `k` trong `t`.
4. **Trả về ký tự bị dịch xa nhất** và khoảng cách.
5. **Isomorphic Strings (205)**, **Find Anagram Mappings (760)**, **Minimum Number of Moves to Make Palindrome (2193)**.

In [26]:
def squared_difference(s, t):
    pos_t = {c: i for i, c in enumerate(t)}
    return sum((i - pos_t[c]) ** 2 for i, c in enumerate(s))


def farthest_char(s, t):
    pos_t = {c: i for i, c in enumerate(t)}
    return max(((abs(i - pos_t[c]), c) for i, c in enumerate(s)))


def with_repeats(s, t):
    """Ghép lần xuất hiện thứ k của ký tự trong s với lần thứ k trong t."""
    from collections import defaultdict
    pos_s, pos_t = defaultdict(list), defaultdict(list)
    for i, c in enumerate(s):
        pos_s[c].append(i)
    for i, c in enumerate(t):
        pos_t[c].append(i)
    return sum(abs(a - b) for c in pos_s for a, b in zip(pos_s[c], pos_t[c]))


print(squared_difference("abc", "bac"))             # 1 + 1 + 0 = 2
print(farthest_char("abcde", "edbac"))              # (4, 'e')
print(with_repeats("aab", "baa"))                   # a: (0,1),(1,2) -> 1 + 1; b: (2,0) -> 2; tổng 4
assert squared_difference("abc", "bac") == 2
assert farthest_char("abcde", "edbac") == (4, "e")
assert with_repeats("aab", "baa") == 4
assert with_repeats("abc", "bac") == 2

2
(4, 'e')
4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `index` | hỏi vị trí mỗi lần | `O(n²)` | `O(1)` | bám sát đề, `n` nhỏ |
| 2. Một `dict` | ghi vị trí trong `t` rồi tra | `O(n)` | `O(n)` | **nên nộp** |
| 3. Hai `dict` | `pos_s`, `pos_t` | `O(n)` | `O(n)` | khi cần cả hai bảng vị trí |
| 4. Mảng 26 | `ord(c) - ord('a')` | `O(n)` | `O(1)` | bảng chữ cái nhỏ, gọn bộ nhớ |
| 5. Hai vòng lồng | tìm tay vị trí trong `t` | `O(n²)` | `O(1)` | học cơ chế của `index` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"hello".index("l")`, `{c: i for i, c in enumerate("xyz")}`, `abs(2 - 5)`, `Solution().findPermutationDifference("ab", "ba")`, `Solution().findPermutationDifference("abc", "cba")`.

**Bài 2 (tính tay).** Với `s = "abcd"` và `t = "dacb"`: lập bảng vị trí của từng ký tự trong `s` và `t`, `|hiệu|` và tổng.

**Bài 3 (điền TODO).** Hoàn thành `permutation_difference_todo(s, t)` theo Cách 2.

In [27]:
def permutation_difference_todo(s, t):
    # TODO 1: tạo dict pos_t: ký tự -> vị trí trong t (dùng enumerate)
    total = 0
    # TODO 2: duyệt (i, c) của s: total += abs(i - pos_t[c])
    # TODO 3: trả về total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `permutation_difference_array2(s, t)` theo Cách 4 **mà không dùng `ord`**: dùng `string.ascii_lowercase.index(c)` để lấy chỉ số chữ cái.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_inversions(s, t)` đếm **số cặp ký tự** `(x, y)` mà thứ tự tương đối của chúng **khác nhau** giữa `s` và `t`. (b) Viết `is_valid_input(s, t)` kiểm tra đầu vào hợp lệ của đề (mỗi ký tự trong `s` chỉ một lần, `t` là hoán vị của `s`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2` (`"hello".index("l")` là vị trí đầu tiên của `l`), `{'x': 0, 'y': 1, 'z': 2}`, `3`, `2`, `4`. Bài 3: dict comprehension `{c: i for i, c in enumerate(t)}`.
- **Mức 2 (vừa):** Bài 2: `a`: `0` và `1` ⇒ `1`; `b`: `1` và `3` ⇒ `2`; `c`: `2` và `2` ⇒ `0`; `d`: `3` và `0` ⇒ `3`; tổng `6`. Bài 4: `idx = string.ascii_lowercase.index(c)`.
- **Mức 3 (sâu):** Bài 5(a): với mọi cặp chữ cái `(x, y)` của `s` có `pos_s[x] < pos_s[y]`, kiểm tra `pos_t[x] > pos_t[y]`. 5(b): `len(set(s)) == len(s)` và `sorted(s) == sorted(t)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "hello".index("l") == 2
assert {c: i for i, c in enumerate("xyz")} == {"x": 0, "y": 1, "z": 2}
assert abs(2 - 5) == 3
assert Solution().findPermutationDifference("ab", "ba") == 2
assert Solution().findPermutationDifference("abc", "cba") == 4

# Bài 2
assert Solution().findPermutationDifference("abcd", "dacb") == 1 + 2 + 0 + 3


# Bài 3
def permutation_difference_todo_ans(s, t):
    pos_t = {c: i for i, c in enumerate(t)}
    total = 0
    for i, c in enumerate(s):
        total += abs(i - pos_t[c])
    return total


for s, t, expected in examples:
    assert permutation_difference_todo_ans(s, t) == expected


# Bài 4
def permutation_difference_array2(s, t):
    pos = [0] * 26
    for i, c in enumerate(t):
        pos[string.ascii_lowercase.index(c)] = i
    return sum(abs(i - pos[string.ascii_lowercase.index(c)]) for i, c in enumerate(s))


for s, t, expected in examples:
    assert permutation_difference_array2(s, t) == expected


# Bài 5(a)
def count_inversions(s, t):
    pos_s = {c: i for i, c in enumerate(s)}
    pos_t = {c: i for i, c in enumerate(t)}
    count = 0
    for x, y in itertools.combinations(s, 2):               # x đứng trước y trong s
        if pos_t[x] > pos_t[y]:
            count += 1
    return count


assert count_inversions("abc", "bac") == 1
assert count_inversions("abc", "cba") == 3
assert count_inversions("abc", "abc") == 0


# Bài 5(b)
def is_valid_input(s, t):
    return len(set(s)) == len(s) and sorted(s) == sorted(t)


assert is_valid_input("abc", "bca") is True
assert is_valid_input("aab", "aba") is False
assert is_valid_input("abc", "abd") is False
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng `|vị trí trong s − vị trí trong t|` của từng ký tự)

```python
class Solution:
    def findPermutationDifference(self, s: str, t: str) -> int:
        pos_t = {c: i for i, c in enumerate(t)}
        return sum(abs(i - pos_t[c]) for i, c in enumerate(s))
```

**Checklist**

- [ ] So sánh vị trí của **cùng một ký tự** trong `s` và `t` (không so `s[i]` với `t[i]`)?
- [ ] Có `abs` cho **từng** ký tự trước khi cộng?
- [ ] Mỗi ký tự được cộng **đúng một lần** (duyệt một chuỗi)?
- [ ] Dùng `dict`/mảng để tra vị trí nhanh (hoặc `index` nếu `n` nhỏ và ký tự không lặp)?
- [ ] Đã thử `"abc"`/`"bac"` (`2`), `"abcde"`/`"edbac"` (`12`), và `s == t` (`0`)?
- [ ] Skeleton đúng `class Solution` / `def findPermutationDifference(self, s: str, t: str) -> int`?